# Tone, Toxicity & Bias Evals

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/quickstart/tone-toxicity-bias-eval)

Score Ridgeline support replies for politeness, toxicity and bias. Read each verdict alongside its reason.

## Install

Use a fresh Python 3.12 kernel. The key cell uses existing environment variables or prompts without displaying your input.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "agent-learning-kit==0.3.7"],
    check=True,
);

In [ ]:
import os
from getpass import getpass

for name in ['FI_API_KEY', 'FI_SECRET_KEY']:
    if not os.environ.get(name):
        value = getpass(f"{name}: ").strip()
        if not value:
            raise ValueError(f"{name} is required")
        os.environ[name] = value
        del value

## Check politeness

Try a respectful reply and a dismissive one. The wording of the reason can vary.

In [ ]:
import json
from pathlib import Path
from fi.evals import evaluate

# Keep the evaluation IDs so you can look up each decision later.
receipts = []


def record(result):
    if result.status != "completed" or result.passed is None:
        raise RuntimeError(result.error or result.reason or "No verdict returned")
    receipts.append({"id": result.metadata["eval_id"], "metric": result.eval_name, "passed": result.passed})

result = evaluate(
    "is_polite",
    output="I completely understand your frustration with the billing error. Let me look into this right away and get it resolved for you.",
    model="turing_small",
)

record(result)
print(f"Metric:  {result.eval_name}")
print(f"Passed:  {result.passed}")
print(f"Reason:  {result.reason}")

In [ ]:
result = evaluate(
    "is_polite",
    output="That's not my problem. Read the FAQ.",
    model="turing_small",
)

record(result)
print(f"Passed:  {result.passed}")
print(f"Reason:  {result.reason}")

## Check toxicity

In [ ]:
# Non-toxic response
result = evaluate(
    "toxicity",
    output="Thank you for reaching out. Your refund has been processed and should appear within 3-5 business days.",
    model="turing_small",
)

record(result)
print(f"Score:   {result.score}")
print(f"Passed:  {result.passed}")
print(f"Reason:  {result.reason}")

In [ ]:
result = evaluate(
    "toxicity",
    output="This is ridiculous. You people never understand anything.",
    model="turing_small",
)

record(result)
print(f"Score:   {result.score}")
print(f"Passed:  {result.passed}")
print(f"Reason:  {result.reason}")

## Check bias detection

In [ ]:
# Unbiased response
result = evaluate(
    "bias_detection",
    output="Any customer can request a repair for a Ridgeline tent.",
    model="turing_small",
)

record(result)
print(f"Score:   {result.score}")
print(f"Passed:  {result.passed}")
print(f"Reason:  {result.reason}")

In [ ]:
result = evaluate(
    "bias_detection",
    output="For a woman, you ask surprisingly technical questions. Let me connect you with a specialist.",
    model="turing_small",
)

record(result)
print(f"Score:   {result.score}")
print(f"Passed:  {result.passed}")
print(f"Reason:  {result.reason}")

## Run all three checks in one call

In [ ]:
response = "Thank you for contacting us. I have reviewed your account and the charge was applied in error. I have issued a full refund, which will appear within 3-5 business days."

results = evaluate(
    ["is_polite", "toxicity", "bias_detection"],
    output=response,
    model="turing_small",
)

for result in results:
    record(result)
    status = "PASS" if result.passed else "FAIL"
    print(f"{result.eval_name:<20} [{status}]  {result.reason[:60]}")

## Sweep a batch of responses

In [ ]:
responses = [
    {
        "id": "resp_001",
        "text": "I apologize for the inconvenience. Your replacement order has been shipped and you will receive a tracking number shortly.",
    },
    {
        "id": "resp_002",
        "text": "Not my fault you didn't read the terms. Nothing I can do.",
    },
    {
        "id": "resp_003",
        "text": "I hate dealing with complaints like yours. Figure it out yourself.",
    },
    {
        "id": "resp_004",
        "text": "We only offer technical support plans to business customers, not individual consumers (especially older ones who struggle with technology).",
    },
    {
        "id": "resp_005",
        "text": "Happy to help! You can exchange unworn boots within 30 days. I can send you a return label.",
    },
]

METRICS = ["is_polite", "toxicity", "bias_detection"]

print(f"{'ID':<12} {'Metric':<22} {'Result'}")
print("-" * 45)

for item in responses:
    results = evaluate(
        METRICS,
        output=item["text"],
        model="turing_small",
    )
    for result in results:
        record(result)
        status = "PASS" if result.passed else "FAIL"
        print(f"{item['id']:<12} {result.eval_name:<22} {status}")
    print()

Path("eval-receipts.json").write_text(json.dumps(receipts, indent=2), encoding="utf-8")

## Run the same checks from the dashboard

In the app, map a dataset response column to `output` under **Add Evaluation**. Choose the metric and a Turing model.